In [1]:
import os
os.chdir(r'C:\Users\Klara\retail-intelligence')

import pandas as pd
import plotly.express as px

results = pd.read_csv('data/processed/baseline_comparison.csv')

print(f"Total products evaluated: {len(results)}")
print(f"Prophet beats naive baseline: {results['BeatNaive'].sum()}/{len(results)}")
print(f"Prophet beats seasonal naive: {results['BeatSeasonalNaive'].sum()}/{len(results)}")

reliable = results[results['DataQualityFlag'] == 'OK']
print(f"\nOK products only ({len(reliable)}):")
print(f"Prophet beats naive baseline: {reliable['BeatNaive'].sum()}/{len(reliable)}")
print(f"Prophet beats seasonal naive: {reliable['BeatSeasonalNaive'].sum()}/{len(reliable)}")

Total products evaluated: 20
Prophet beats naive baseline: 7/20
Prophet beats seasonal naive: 13/20

OK products only (18):
Prophet beats naive baseline: 7/18
Prophet beats seasonal naive: 13/18


In [2]:
print("Average MAE — all 20 products:")
print(f"  Prophet MAE:          {results['Prophet_MAE'].mean():.1f}")
print(f"  Naive MAE:            {results['Naive_MAE'].mean():.1f}")
print(f"  Seasonal Naive MAE:   {results['SeasonalNaive_MAE'].mean():.1f}")

print("\nAverage MAE — 18 OK products only:")
print(f"  Prophet MAE:          {reliable['Prophet_MAE'].mean():.1f}")
print(f"  Naive MAE:            {reliable['Naive_MAE'].mean():.1f}")
print(f"  Seasonal Naive MAE:   {reliable['SeasonalNaive_MAE'].mean():.1f}")

metrics_plot = pd.DataFrame({
    'Model': ['Prophet', 'Naive Baseline', 'Seasonal Naive'] * 2,
    'Mean MAE': [
        results['Prophet_MAE'].mean(), results['Naive_MAE'].mean(), results['SeasonalNaive_MAE'].mean(),
        reliable['Prophet_MAE'].mean(), reliable['Naive_MAE'].mean(), reliable['SeasonalNaive_MAE'].mean()
    ],
    'Product Set': ['All 20 products'] * 3 + ['OK products only (18)'] * 3
})

fig = px.bar(
    metrics_plot,
    x='Model',
    y='Mean MAE',
    color='Product Set',
    barmode='group',
    title='Average MAE: Prophet vs Baselines — All Products vs OK-Only'
)
fig.show()

Average MAE — all 20 products:
  Prophet MAE:          1002.0
  Naive MAE:            851.0
  Seasonal Naive MAE:   1003.0

Average MAE — 18 OK products only:
  Prophet MAE:          336.1
  Naive MAE:            379.5
  Seasonal Naive MAE:   546.4


In [3]:
reliable_plot = reliable.copy()
reliable_plot['MAE_Diff'] = reliable_plot['Naive_MAE'] - reliable_plot['Prophet_MAE']
reliable_plot = reliable_plot.sort_values('MAE_Diff', ascending=False)

fig2 = px.bar(
    reliable_plot,
    x='StockCode',
    y='MAE_Diff',
    color='BeatNaive',
    title='Prophet Advantage Over Naive by Product (Naive MAE − Prophet MAE)',
    labels={'MAE_Diff': 'MAE improvement (positive = Prophet better)'},
    color_discrete_map={True: '#1f77b4', False: '#d62728'}
)
fig2.add_hline(y=0, line_dash='dash', line_color='gray')
fig2.show()

In [4]:
demand = pd.read_csv('data/processed/weekly_demand.csv')
demand['Week'] = pd.to_datetime(demand['Week'])

product_21977 = demand[demand['StockCode'] == '21977'].sort_values('Week')

print(product_21977[['Week', 'TotalQuantity']].tail(10))
print(f"\nLast 8 weeks (the test set):")
print(product_21977['TotalQuantity'].tail(8).describe())

          Week  TotalQuantity
638 2011-10-03            827
639 2011-10-10            146
640 2011-10-17            111
641 2011-10-24             95
642 2011-10-31             58
643 2011-11-07            145
644 2011-11-14            122
645 2011-11-21            357
646 2011-11-28            230
647 2011-12-05            170

Last 8 weeks (the test set):
count      8.000000
mean     161.000000
std       94.491118
min       58.000000
25%      107.000000
50%      133.500000
75%      185.000000
max      357.000000
Name: TotalQuantity, dtype: float64


In [5]:
forecasts = pd.read_csv('data/processed/all_forecasts.csv')
forecasts['ds'] = pd.to_datetime(forecasts['ds'])

prophet_21977 = forecasts[
    (forecasts['StockCode'] == '21977') & 
    (forecasts['ds'] >= '2011-10-10') & 
    (forecasts['ds'] <= '2011-12-05')
]
print(prophet_21977[['ds', 'yhat']])

            ds        yhat
727 2011-10-10  384.343776
728 2011-10-17  385.171299
729 2011-10-24  385.998822
730 2011-10-31  386.826345
731 2011-11-07  387.653868
732 2011-11-14  388.481391
733 2011-11-21  389.308914
734 2011-11-28  390.136437
735 2011-12-05  390.963960


## Day 4 — Notes & Observations

### What each cell did

**Data loading cell** — loaded `baseline_comparison.csv`, comparing win-rates across all 20 products vs. the 18 "OK" (non-flagged) products. Both give the same win-rate: Prophet beats naive on 7/18-20 products, and beats seasonal naive on 13/18-20 products.

**Average MAE comparison** — confirmed the earlier script's finding: including the 2 flagged products inflates Prophet's average MAE from 336.1 (OK only) to 1002.0 (all 20), nearly 3x. The flagged products' extreme, unreliable forecasts distort any simple average - a strong argument for always reporting metrics both ways (with and without flagged products) rather than a single blended number.

**Per-product MAE improvement chart** — visualizes the win/loss pattern directly: 
Prophet wins by a wide margin on a handful of products (22197: +540 units, 84879, 84077, 22616), while losing by a narrow margin on most others, except one clear outlier: 21977 (-150), investigated below.

### Key finding: Prophet wins on average, loses on the median product

This seems contradictory at first but isn't: Prophet's lower *average* MAE (336.1 vs Naive's 379.5) comes entirely from large wins on a minority of products (7/18), not from being more accurate than naive on a typical product. On the majority of products (11/18), naive - "just repeat last week's actual value" - produces a slightly better forecast than Prophet's trend-only model.

This is an honest, not-entirely-flattering finding about Prophet in this specific setup (trend-only, due to the yearly/weekly seasonality limitations from Days 1-2). It doesn't mean Prophet is a bad choice - the seasonal naive comparison (13/18 wins) is more favorable, and Prophet's failures are all small margins while its wins are large - but a simple "Prophet beats the baseline" headline would overstate the case. Both the average and the win-rate belong in the final evaluation writeup, not just whichever number looks better.

### Why Prophet beats seasonal naive more easily than plain naive

Seasonal naive predicts "same week as last year" - a single specific historical data point, not a validated pattern (same limitation that led to disabling yearly_seasonality in Days 1-2). It's a fundamentally weaker baseline given only ~1 year of history, so Prophet clears it more often (13/18) than it clears plain naive (7/18).

### Sanity check: why does Prophet lose worst on 21977?

Checked directly by pulling actuals vs. Prophet's forecast for the test window:

| Week | Actual | Prophet yhat |
|---|---|---|
| 2011-10-17 | 111 | 385.2 |
| 2011-10-24 | 95 | 386.0 |
| 2011-10-31 | 58 | 386.8 |
| 2011-11-07 | 145 | 387.7 |
| 2011-11-14 | 122 | 388.5 |
| 2011-11-21 | 357 | 389.3 |
| 2011-11-28 | 230 | 390.1 |
| 2011-12-05 | 170 | 391.0 |

Prophet forecasted a nearly flat ~385-391 units/week for all 8 test weeks, while actual demand ranged 58-357 (mean 161). Naive's flat guess of 146 (the last training value) happened to sit much closer to the true range, giving it a large edge.

Cause: 21977 has a "Growing" trend direction (Day 2 analysis). With no seasonality component to temper it, the trend line kept climbing straight through a period where actual demand stayed flat/low - a genuine limitation of a trend-only model, distinct from the sparse-data DataQualityFlag issue. A product can be "OK" by that flag's criteria and still get a bad forecast purely from trend overshoot.

**Not fixed today.** This is a single 8-week snapshot, and tuning `changepoint_prior_scale` or switching to logistic growth against one holdout 
risks overfitting the choice to this particular window. Deferred to Week 5's walk-forward validation, which will show whether this overshoot is a systematic pattern across growing-trend products or specific to this test split. Candidate fixes once there's more evidence: reduce `changepoint_prior_scale`, or use logistic growth with a demand cap.

### Caveat: this is a simplified comparison, not yet rigorous

Per the original Week 4 plan, this comparison holds out the last 8 weeks as a single test split without refitting - full walk-forward validation (multiple rolling train/test windows) comes in Week 5. Today's numbers, including the 21977 finding above, are a first pass and may shift once validated across multiple time windows rather than one.